In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set before torch uses the GPU

import torch
import gc
import time
import pandas as pd

from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

In [2]:
# log in to hugging face to access gated models (llama)

login(UserSecretsClient().get_secret("HF_TOKEN"))

In [3]:
# test for GPU 

print(torch.cuda.is_available())   # should print True

True


#### Data Preparation
- load pool.csv and convert from csv to pandas df.
- cloen github repo to access files offloaded onto repo, get llm prompt messages.

In [4]:
# import csv file for financialphrasebank pool split

pool = pd.read_csv('/kaggle/input/datasets/thisisbeancurd/quorum-pool-test-ds/pool.csv')
print(pool.head())
print(f"\nPool Set Size: {len(pool)}")

            item_id                                           sentence  label
0  d05f057d78da5246  Finnish flexible packaging manufacturer Suomin...      2
1  63834bfeeba4fbbd               No financial detail were available .      1
2  5beff7676d10a84c  RK Group , headquartered in Vantaa , Finland ,...      1
3  b11995c7c292c6b3  The contract covers turnkey deliveries to all ...      1
4  fe26c12e86082bda  The total capacity of the factory will be appr...      1

Pool Set Size: 4112


In [5]:
# glone my github repo to access repo files

!git clone https://github.com/thisisadison/quorum

Cloning into 'quorum'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 82 (delta 35), reused 62 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 1.29 MiB | 6.71 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [6]:
PROMPT_V1 = open("quorum/configs/prompts/prompt_v1.txt").read()
print(PROMPT_V1)

ROLE:
You are a financial sentiment classifier trained to assess how financial
news affects investor perception of a company.

INSTRUCTION:
Classify the sentiment of the given sentence, from the perspective of an
investor evaluating its effect on the company's stock outlook.

CONTEXT:
The sentence is drawn from a financial news article. It may describe
company earnings, operations, strategy, or market events.

CONSTRAINTS:
Base your judgment only on the text provided. Do not assume information
not stated. Do not explain your reasoning.

OUTPUT FORMAT:
Respond with exactly one word: positive, negative, or neutral.
No punctuation, no additional text.

Sentence: {text}


#### Qwen3-8B

- import qwen, define sample sentence, prompt and tokenize input message.
- add_generation_prompt and enable_thinking disabled as do not need llm chat.
- extract probabilities for each label predicted, perform check to ensure accurate reflection of label probability distribution.

In [7]:
model_name = "Qwen/Qwen3-8B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

# prepare the model input
sentence = "For the first nine months of 2010 , Talvivaara 's net loss narrowed to EUR 8.3 million from EUR 21.9 million for the same period of 2009 ."

prompt = PROMPT_V1.format(text=sentence)
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [8]:
LABELS = ["negative", "neutral", "positive"]

# 1. one forward pass: collect scores
with torch.no_grad():
    outputs = model(**model_inputs)

# 2. take the scores for the NEXT word = the first word of the answer
next_logits = outputs.logits[0, -1]

# 3. turn scores into probabilities over the whole vocabulary
probs = torch.softmax(next_logits.float(), dim=-1)

# 4. find the token id for each label word
label_ids = {lab: tokenizer.encode(lab, add_special_tokens=False)[0] for lab in LABELS}

# 5. print each label's probability
for lab, tid in label_ids.items():
    print(f"{lab:9} token={tokenizer.decode([tid])!r:12} p={probs[tid].item():.4f}")

total = sum(probs[tid].item() for tid in label_ids.values())
print(f"\nsum of 3 labels: {total:.4f}")

negative  token='negative'   p=0.0003
neutral   token='neutral'    p=0.0003
positive  token='positive'   p=0.9995

sum of 3 labels: 1.0000


#### Label Probability Functions

- read each label's probability from the first token of the model's answer, summing all forms of a label (positive, Positive, ' positive').
- flag sentences where the 3 labels get under 90% of the probability, stop if more than 1% are flagged or any are NaN.
- rescale the 3 probabilities to add up to 1, take the highest as the llm label.

In [9]:
LABELS = ["negative", "neutral", "positive"]
LOW_MASS = 0.90 # probability threshold for all 3 labels combined
MAX_LOW_SHARE = 0.01 # percentage of sentences flagged

# every way a label can be represented in the vocabulary
LABEL_FORMS = {
    "negative": ["negative", "Negative", " negative", " Negative"],
    "neutral":  ["neutral",  "Neutral",  " neutral",  " Neutral"],
    "positive": ["positive", "Positive", " positive", " Positive"],
}


def get_label_token_ids(tokenizer):
    """token ids of every form of each label (e.g. positive, Positive, ' positive')"""
    label_ids = {}
    for lab, forms in LABEL_FORMS.items():
        ids = set()
        for form in forms:
            tokens = tokenizer.encode(form, add_special_tokens=False)
            # some tokenizers turn a leading space into its own token; skip it
            if tokenizer.decode([tokens[0]]).strip() == "" and len(tokens) > 1:
                ids.add(tokens[1])
            else:
                ids.add(tokens[0])
        label_ids[lab] = ids
    return label_ids


def check_label_tokens(tokenizer):
    """stop if any token would be counted for two different labels"""
    label_ids = get_label_token_ids(tokenizer)
    all_ids = [tid for ids in label_ids.values() for tid in ids]
    assert len(all_ids) == len(set(all_ids)), f"labels share a token: {label_ids}"


def get_label_probs(sentence, prompt_template, model, tokenizer):
    """get probability distribution across the three labels"""
    prompt = prompt_template.format(text=sentence)
    messages = [
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**model_inputs)

    next_logits = outputs.logits[0, -1]
    probs = torch.softmax(next_logits.float(), dim=-1)

    # sum the probability of every form of each label
    label_ids = get_label_token_ids(tokenizer)
    return {lab: sum(probs[tid].item() for tid in ids) for lab, ids in label_ids.items()}


def normalise(probs):
    """rescale the 3 probabilities to add up to 1"""
    mass = sum(probs.values())

    scaled = {}
    for lab in probs:
        scaled[lab] = probs[lab] / mass

    return scaled, mass


def label_dataset(df, prompt_template, model, tokenizer):
    """run every sentence in df through the model. returns one row of results per sentence"""
    results = []
    for _, row in df.iterrows():
        probs = get_label_probs(row["sentence"], prompt_template, model, tokenizer)
        probs, mass = normalise(probs)
        results.append({
            "item_id": row["item_id"],
            **probs,
            "llm_label": max(probs, key=probs.get),
            "label_mass": mass,
            "low_mass": mass < LOW_MASS,
        })
    return pd.DataFrame(results)


def check_label_mass(results):
    """stop if too many sentences put too little probability on the 3 labels"""
    share_low = results["low_mass"].mean()
    print(f"\nLowest label_mass: {results['label_mass'].min():.4f}, Flagged: {share_low:.1%}")
    assert results["label_mass"].notna().all(), "some probabilities are NaN (float16 problem)"
    assert share_low <= MAX_LOW_SHARE, f"{share_low:.1%} of sentences have low label mass"

#### Evaluation Functions

- attach human labels to llm labels by item_id.
- report accuracy, macro-f1, per-class scores and confusion matrix.
- compare confidence when the llm is right vs wrong, save results to /kaggle/working.

In [10]:
# perform evaluation for llm label

LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}


def add_human_labels(results, df):
    """attach the human label to each result, matched on item_id"""
    compare = results.merge(df[["item_id", "label"]], on="item_id")
    compare["human_label"] = compare["label"].map(LABEL_MAP)
    compare["confidence"] = compare[LABELS].max(axis=1)
    compare["correct"] = compare["llm_label"] == compare["human_label"]
    return compare


def evaluate(compare, model_name):
    """print the detailed report and return the key numbers for the comparison table"""
    print(f"\n===== {model_name} =====")
    print(classification_report(compare["human_label"], compare["llm_label"]))
    print(confusion_matrix(compare["human_label"], compare["llm_label"], labels=LABELS))

    return {
        "model": model_name,
        "accuracy": accuracy_score(compare["human_label"], compare["llm_label"]),
        "macro_f1": f1_score(compare["human_label"], compare["llm_label"], average="macro"),
        "below_0.90": (compare["confidence"] < 0.90).sum(),
        "conf_when_right": compare.loc[compare["correct"], "confidence"].mean(),
        "conf_when_wrong": compare.loc[~compare["correct"], "confidence"].mean(),
    }


def save_results(compare, model_name):
    """save file to retain findings after the session"""
    safe_name = model_name.split("/")[-1]        # "Qwen/Qwen3-8B" -> "Qwen3-8B"
    compare.to_csv(f"/kaggle/working/pilot_{safe_name}.csv", index=False)

#### Pilot Runner/ / Model Set-Up

- load model, check label tokens, smoke test on 5 sentences, pilot on 100 sentences.
- check label mass, evaluate, save, then free the GPU even if a check fails.

In [11]:
# load and free models, run the full pilot for one model

def load_model(model_name):
    """load tokenizer and model onto the GPUs"""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    return model, tokenizer


def free_gpu():
    """clear GPU memory so the next model fits, then show what is left"""
    gc.collect()
    torch.cuda.empty_cache()
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 1e9:.1f} GB in use")


def run_pilot(model_name, pilot, prompt_template):
    """smoke test, pilot, checks, evaluation and saving for one model"""
    model, tokenizer = None, None
    try:
        model, tokenizer = load_model(model_name)
        check_label_tokens(tokenizer)

        label_dataset(pilot.head(5), prompt_template, model, tokenizer)   # smoke test

        start = time.time()
        results = label_dataset(pilot, prompt_template, model, tokenizer)
        elapsed = time.time() - start
        print(f"\n{elapsed:.1f}s for {len(pilot)} sentences")

        check_label_mass(results)
        compare = add_human_labels(results, pilot)
        save_results(compare, model_name)

        summary = evaluate(compare, model_name)
        summary["seconds_per_100"] = elapsed / len(pilot) * 100
        return summary
    finally:
        # runs whether the pilot succeeded or a check failed
        del model, tokenizer
        free_gpu()

In [12]:
# perform smoke test on 5 entries before pilot

smoke = pool.sample(5, random_state=0)
print(label_dataset(smoke, PROMPT_V1, model, tokenizer))

            item_id      negative       neutral      positive llm_label  \
0  7de0b5adfec29b50  2.940079e-11  2.845335e-08  1.000000e+00  positive   
1  25dfe807c2dd216e  3.192259e-09  3.763643e-05  9.999624e-01  positive   
2  6b308f6c0cce4ec7  9.999999e-01  1.315653e-07  1.026425e-10  negative   
3  c84871644879e112  6.358387e-12  1.000000e+00  1.402649e-10   neutral   
4  0aa486d50f239b68  2.849621e-11  1.000000e+00  2.413364e-12   neutral   

   label_mass  low_mass  
0         1.0     False  
1         1.0     False  
2         1.0     False  
3         1.0     False  
4         1.0     False  


In [13]:
# perform pilot test on sample of pool set, record latency

check_label_tokens(tokenizer)

pilot = pool.sample(100, random_state=42)

start = time.time()
pilot_results = label_dataset(pilot, PROMPT_V1, model, tokenizer)
elapsed = time.time() - start

print(f"\n{elapsed:.1f}s for 100 sentences. Estimated {elapsed * len(pool) / 100 / 60:.0f} min for full pool.")

check_label_mass(pilot_results)
print(pilot_results.head())


17.6s for 100 sentences. Estimated 12 min for full pool.

Lowest label_mass: 1.0000, Flagged: 0.0%
            item_id      negative       neutral  positive llm_label  \
0  f1aea9bd38e3d81c  5.315738e-08  9.223469e-06  0.999991  positive   
1  7ff8f4626756522b  1.615523e-07  3.629686e-01  0.637031  positive   
2  cd151ec92ef34437  7.123274e-10  1.522957e-08  1.000000  positive   
3  d7227de9dc4db3a3  1.997738e-09  9.999990e-01  0.000001   neutral   
4  6e25829a3588858d  8.046778e-10  9.969246e-01  0.003075   neutral   

   label_mass  low_mass  
0         1.0     False  
1         1.0     False  
2         1.0     False  
3         1.0     False  
4         1.0     False  


In [14]:
# evaluate qwen pilot against human labels and save

compare = add_human_labels(pilot_results, pilot)
save_results(compare, "Qwen/Qwen3-8B")

summaries = []
qwen_summary = evaluate(compare, "Qwen/Qwen3-8B")
qwen_summary["seconds_per_100"] = elapsed / len(pilot) * 100
summaries.append(qwen_summary)


===== Qwen/Qwen3-8B =====
              precision    recall  f1-score   support

    negative       0.57      1.00      0.73         8
     neutral       0.90      0.79      0.84        67
    positive       0.67      0.72      0.69        25

    accuracy                           0.79       100
   macro avg       0.71      0.84      0.75       100
weighted avg       0.81      0.79      0.79       100

[[ 8  0  0]
 [ 5 53  9]
 [ 1  6 18]]


In [15]:
# remove qwen from GPU so the next model fits

del model, tokenizer, outputs, model_inputs
free_gpu()

GPU 0: 0.1 GB in use
GPU 1: 0.0 GB in use


#### Remaining Models

- run the same 100 pilot sentences and prompt through gemma, llama and mistral.
- mistral kept for comparison only, replaced by gemma 
- granite dropped, its answers often start with a tag so the label can't be read from the first token.

In [16]:
# pilot for the remaining models, same 100 sentences and prompt

MODELS = [
    "google/gemma-2-9b-it",                 # largest, so it loads first onto empty GPUs
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",   # kept as evidence for why it was replaced
]

for model_name in MODELS:
    try:
        summaries.append(run_pilot(model_name, pilot, PROMPT_V1))
    except Exception as e:
        print(f"\n!!! {model_name} failed: {e}")
        free_gpu()

print(pd.DataFrame(summaries))

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

... (24 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


21.9s for 100 sentences

Lowest label_mass: 0.9986, Flagged: 0.0%

===== mistralai/Mistral-7B-Instruct-v0.3 =====
              precision    recall  f1-score   support

    negative       0.50      1.00      0.67         8
     neutral       0.96      0.36      0.52        67
    positive       0.39      0.92      0.55        25

    accuracy                           0.55       100
   macro avg       0.62      0.76      0.58       100
weighted avg       0.78      0.55      0.54       100

[[ 8  0  0]
 [ 7 24 36]
 [ 1  1 23]]
GPU 0: 0.1 GB in use
GPU 1: 0.0 GB in use
                                model  accuracy  macro_f1  below_0.90  \
0                       Qwen/Qwen3-8B      0.79  0.753617           6   
1                google/gemma-2-9b-it      0.74  0.737896          11   
2    meta-llama/Llama-3.1-8B-Instruct      0.83  0.780046          29   
3  mistralai/Mistral-7B-Instruct-v0.3      0.55  0.578675           5   

   conf_when_right  conf_when_wrong  seconds_per_100  
0   